<div style="background:linear-gradient(135deg,#1f3a93,#6c3fc5);color:white;padding:22px 26px;border-radius:14px">
<h1 style="margin:0;color:white">🧑‍🔬 Lab 2: How Does a Machine "Recognise" a Face?</h1>
<p style="margin:6px 0 0 0;font-size:16px">An interactive hands-on session on <b>learning, evaluation and the limits of machine "intelligence"</b></p>
</div>


Face recognition for some time looked like magic. Your phone unlocks whether in the day or night, smiling or frowning, well dressed or rough. Underneath, it is **geometry on numbers**. In this workshop you will *play the algorithm by hand*, build a classic recogniser (**eigenfaces + nearest neighbour**) with Python, and then *change one hyperparameter at a time, predict what will happen, and test whether you were right.* We close by asking the philosophical question the maths cannot answer: *does the system "recognise" anything at all?*

## Learning outcomes
By the end you will be able to:
1. Set up and use a Python AI environment (Jupyter, NumPy, scikit-learn, Matplotlib).
2. Explain how a machine-learning model turns raw pixels to useful features and into a decision.
3. Show how hyperparameters (PCA size, *k*, distance metric, threshold…) change accuracy and error types.
4. Use evaluation metrics like accuracy, false accept rate (FAR), false reject rate (FRR), and why one number is never enough.
5. Distinguish optimal/rational behaviour from human-like behaviour, and discuss ethics, bias and robustness.

## How to use this notebook
| Icon | Meaning |
|---|---|
| 🎛️ | **Interactive**: sliders and toggles. Change them and watch the output update |
| 🔮 | **Predict first**: write your guess *before* you move a slider (this is the scientific method) |
| 🆘 | **Hint**: click to expand, but try first |
| ✅ | **Solution / explanation**: click to expand |
| 🤔 | **Discuss** with your neighbour |
| ⚠️ | **Pitfall**: a common mistake |
| 🧠 | **Quick quiz** |


> 💬 **Run cells top to bottom** (`Shift+Enter`). If a slider does nothing, make sure the cell above it has been run.

# Environment setup (6 min)

### 🛠️ Setup options
- **Easiest:** Google Colab: upload this notebook; everything is preinstalled.
- **Local:** `pip install numpy scikit-learn matplotlib ipywidgets jupyterlab`, then run `jupyter lab`.
- **Tip:** use a virtual environment (`python -m venv .venv`) so projects do not clash.

Run the check below. Every line should print a version.

In [ ]:
import sys, time
import numpy as np
import sklearn, matplotlib
import matplotlib.pyplot as plt

print("Python       :", sys.version.split()[0])
print("NumPy        :", np.__version__)
print("scikit-learn :", sklearn.__version__)
print("Matplotlib   :", matplotlib.__version__)
try:
    import ipywidgets as W
    print("ipywidgets   :", W.__version__)
except ImportError:
    raise SystemExit("❌ ipywidgets missing. Run:  pip install ipywidgets   (then restart the kernel)")
assert sys.version_info >= (3, 9), "Please use Python 3.9 or newer"
print("\n✅ Environment OK. Welcome to the lab!")

🆘 <details><summary><b>Hint: something failed?</b></summary>

- `ModuleNotFoundError` → install the package: `pip install <name>` (in Colab put `!` in front), then **restart the kernel** (Kernel → Restart).
- Sliders show as text like `IntSlider(value=…)` → the widgets extension is not active. Use **Colab** or **JupyterLab ≥ 4**, or run `pip install -U ipywidgets jupyterlab`.
- Corporate/campus network blocks the dataset download in Block 3 → ask the instructor for the cached file, or use Colab.
</details>

In [ ]:
# An image is just a grid of numbers.
tiny_face = np.array([[0, 0, 9, 0, 0],
                      [0, 9, 0, 9, 0],
                      [0, 0, 0, 0, 0],
                      [9, 0, 0, 0, 9],
                      [0, 9, 9, 9, 0]])
print("shape:", tiny_face.shape, "| brightest pixel:", tiny_face.max(), "| mean:", tiny_face.mean().round(2))
plt.figure(figsize=(2.2, 2.2)); plt.imshow(tiny_face, cmap="gray_r"); plt.axis("off"); plt.title("a 5×5 face"); plt.show()
print("flattened to a vector of", tiny_face.ravel().size, "numbers:", tiny_face.ravel()[:10], "…")

🧠 **Quick quiz**: a 64×64 greyscale photo, flattened, becomes a vector of how many numbers?

In [ ]:
from IPython.display import display, HTML, clear_output

def quiz(question, options, answer, why):
    '''Tiny multiple-choice widget: quiz("Q?", ["a","b","c"], answer_index, "explanation")'''
    rb = W.RadioButtons(options=options, value=None, layout=W.Layout(width="auto"))
    btn = W.Button(description="Check ✔️", button_style="info")
    out = W.Output()
    def check(_):
        with out:
            clear_output()
            if rb.value is None:
                print("👆 Pick an answer first.")
            elif rb.value == options[answer]:
                print("✅ Correct!", why)
            else:
                print("❌ Not quite. Think again and re-check (🆘 re-read the concept box).")
    btn.on_click(check)
    display(W.VBox([W.HTML(f"<b>🧠 {question}</b>"), rb, btn, out]))

quiz("A 64×64 greyscale photo, flattened, becomes a vector of how many numbers?",
     ["64", "128", "4,096", "64,000"], 2,
     "64 × 64 = 4,096 pixel values, so each photo is one point in a 4,096-dimensional space.")

### Block 1: Concept primer (2 min read)

Two different questions
| Task | Question | Answer type | Example |
|---|---|---|---|
| **Verification** (1 : 1) | "Are you who you claim to be?" | yes / no | phone unlock |
| **Identification** (1 : N) | "Who are you, out of everyone I know?" | a name, **or "unknown"** | border gate, photo tagging |

Today we build **identification**, and in Block 8 we add the *"unknown"* option (called **open-set** recognition).


The PEAS description of our recogniser
Russell & Norvig describe a **rational agent** by its **P**erformance measure, **E**nvironment, **A**ctuators, **S**ensors.
| PEAS | Face recogniser |
|---|---|
| **P**erformance | few false accepts and few false rejects (*we choose this, it is a value judgement*) |
| **E**nvironment | enrolled database, lighting, camera angle, people with/without glasses or masks |
| **A**ctuators | unlock / reject / flag for human review |
| **S**ensors | camera |

> 🤔 **Human-like or rational?** The system does *not* mimic how people recognise faces (context, voices, gait, memory). It just minimises a distance to maximise its performance measure, which is **rational-agent behaviour, not human-like behaviour**. Keep this in mind for the debrief.

### Block 2: Be the algorithm (9 min)

The system never "sees" a face. It **measures** it and compares the measurements to people it has already **enrolled**.
Below, 4 people are enrolled with 3 measurements each (in mm). A new person (the **probe**) stands in front of the camera.

#### ✋ Step A: by hand (pairs, 4 min)
Probe = `[61, 32, 114]`. For each enrolled person compute

$$d = \sqrt{(\text{eye}_1-\text{eye}_2)^2+(\text{nose}_1-\text{nose}_2)^2+(\text{jaw}_1-\text{jaw}_2)^2}$$

**Predict:** who will be closest? _(write your guess on paper)_

<details><summary>🆘 Hint</summary>

Do Chi first: differences are `(61−60, 32−31, 114−112) = (1, 1, 2)` → `√(1+1+4) = √6 ≈ 2.45`.
</details>

In [ ]:
features = ["eye_distance", "nose_width", "jaw_width"]
gallery = {                       # 🗄️ the enrolled "database"
    "Ada":  np.array([62, 34, 118]),
    "Ben":  np.array([66, 38, 130]),
    "Chi":  np.array([60, 31, 112]),
    "Dayo": np.array([68, 40, 124]),
}
probe = np.array([61, 32, 114])   # 🧍 the person at the camera

for n, v in gallery.items(): print(f"{n:5s} {v}")
print("PROBE", probe)

# 🔢 Step B: check your hand calculation with NumPy
dist = {n: np.linalg.norm(probe - v) for n, v in gallery.items()}
for n, d in sorted(dist.items(), key=lambda kv: kv[1]): print(f"{n:5s} distance = {d:5.2f}")
print("\n🏁 Closest match:", min(dist, key=dist.get))

#### 🎛️ Step C: interactive toy recogniser
Move the probe's measurements, change the **threshold** (reject if even the best match is too far), toggle **standardise** and the **distance metric**.

**What to look for**
- 🔮 *Predict:* what happens to the decision when you set the threshold very low? very high?
- 🔮 *Predict:* does standardising (rescaling each feature to a similar range) change who is closest?
- ⚠️ *Pitfall:* raw features have different scales (jaw ≈ 110–130, nose ≈ 30–40). A feature with big numbers can *dominate* the distance without being more informative.

<details><summary>🆘 Hint: scenarios to try</summary>

- 🕵️ **Stranger:** eye=75, nose=45, jaw=140 with threshold 10 → should be rejected.
- 🎭 **Look-alike:** eye=64, nose=36, jaw=121 → is the match confident? compare the top two bars.
- When *standardise* is on, distances are measured in "standard deviations", so sensible thresholds are about 0.5–3, not 10.
</details>

In [ ]:
import matplotlib.patches as mpatches

names = list(gallery)
G = np.array([gallery[n] for n in names], float)

def toy_lab(eye=61, nose=32, jaw=114, threshold=10.0, standardise=False, metric="euclidean"):
    p = np.array([eye, nose, jaw], float)
    Gx = G.copy()

    if standardise:
        mu, sd = G.mean(0), G.std(0)
        Gx, p = (G - mu) / sd, (p - mu) / sd
    diff = Gx - p
    d = np.linalg.norm(diff, axis=1) if metric == "euclidean" else np.abs(diff).sum(1)
    best = int(np.argmin(d))
    accepted = d[best] <= threshold

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.3, 1]})
    cols = ["#2e8b57" if (i == best and accepted) else "#999" for i in range(len(names))]
    ax[0].bar(names, d, color=cols)
    ax[0].axhline(threshold, color="crimson", ls="--", label=f"threshold = {threshold}")
    ax[0].set_ylabel(f"distance ({metric})")
    ax[0].set_title("Distance from probe to each enrolled person")
    ax[0].legend()
    contrib = (np.abs(diff[best]) if metric == "manhattan" else diff[best] ** 2)
    contrib = contrib / contrib.sum() if contrib.sum() else contrib
    ax[1].barh(features, contrib, color="#6c3fc5")
    ax[1].set_xlim(0, 1)
    ax[1].invert_yaxis()
    ax[1].set_title(f"Which feature drives the match to {names[best]}?")
    plt.tight_layout()
    plt.show()

    srt = np.sort(d)
    msg = f"✅ ACCEPT as <b>{names[best]}</b>" if accepted else "🚫 REJECT: <b>UNKNOWN</b>"
    print_html = f"<div style='font-size:17px;padding:8px'>{msg} &nbsp;·&nbsp; best distance {srt[0]:.2f}, runner-up {srt[1]:.2f} (margin {srt[1]-srt[0]:.2f})</div>"
    display(HTML(print_html))

W.interact(toy_lab,
    eye=W.IntSlider(61, min=50, max=80, description="👁️ eye", continuous_update=False),
    nose=W.IntSlider(32, min=25, max=50, description="👃 nose", continuous_update=False),
    jaw=W.IntSlider(114, min=100, max=145, description="🦴 jaw", continuous_update=False),
    threshold=W.FloatSlider(10, min=0.5, max=40, step=0.5, description="🚦 threshold", continuous_update=False),
    standardise=W.Checkbox(False, description="standardise features"),
    metric=W.ToggleButtons(options=["euclidean", "manhattan"], description="metric"))

### 🤔 Discuss (2 min)
1. When does a **low** threshold hurt? A **high** one? (These are *false rejects* and *false accepts*. Remember the words.)
2. Why might *standardising* change the winner?
3. **Challenge 🕹️:** find a probe that the system confidently labels as the **wrong** person. What does this say about "recognition"?

<details><summary>✅ Explanation</summary>

- Low threshold → even the genuine person is turned away (**false reject**). High threshold → strangers get through (**false accept**).
- Without standardising, the jaw (largest numbers) contributes most of the distance, so the model effectively "listens" mostly to jaw width. Standardising gives each feature an equal voice.
- A wrong-but-confident match shows the model has no concept of *identity*, only of *distance in feature space*. That is the heart of the human-like vs rational distinction.
</details>

In [ ]:
quiz("A system lets a stranger in because its threshold is too generous. This is a…",
     ["False reject", "False accept", "True accept", "True reject"], 1,
     "The stranger should have been rejected but was accepted: a false accept (a security risk).")

# Block 3: Real data (3 min)

**Olivetti faces** (a.k.a. ORL): 400 greyscale photos, **40 people × 10 photos**, 64×64 pixels, taken at AT&T Laboratories Cambridge in the early 1990s with varying expression, glasses and lighting.
> ⚠️ **Pitfall:** it is tiny, old and not representative of the real world. We use it because it is small enough to experiment with in seconds, **not** because results here would transfer to a real deployment.

train / test split
- **Train**: what the model may look at.
- **Test**: new photos of the *same people*, hidden during training. Testing on training photos would be like marking an exam with the questions the student memorised.

*(First run needs internet for a few MB.)*

In [ ]:
from functools import lru_cache
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.metrics import accuracy_score

faces = fetch_olivetti_faces(shuffle=True, random_state=0)
X, y, IMG = faces.data, faces.target, faces.images
print(f"📦 {IMG.shape[0]} images of {IMG.shape[1]}×{IMG.shape[2]} px, flattened {X.shape} | {len(np.unique(y))} people × {np.bincount(y)[0]} photos")

def browse(person=0):
    idx = np.where(y == person)[0]
    fig, ax = plt.subplots(1, len(idx), figsize=(1.5 * len(idx), 1.8))
    for a, i in zip(np.atleast_1d(ax), idx):
      a.imshow(IMG[i], cmap="gray")
      a.axis("off")
    plt.suptitle(f"👤 person {person}: {len(idx)} photos. Spot the variation (expression, glasses, angle)", y=1.02)
    plt.show()

W.interact(browse, person=W.IntSlider(0, min=0, max=39, description="👤 person", continuous_update=False));

In [ ]:
# 🔀 Split helpers (cached so the sliders stay fast)
@lru_cache(maxsize=64)
def get_split(test_size, seed):
    idx = np.arange(len(y))
    tr, te = train_test_split(idx, test_size=test_size, stratify=y, random_state=seed)
    return tr, te

@lru_cache(maxsize=256)
def fit_pca(test_size, seed, n_components, whiten):
    tr, _ = get_split(test_size, seed)
    return PCA(n_components=n_components, whiten=whiten, random_state=0).fit(X[tr])

tr, te = get_split(0.25, 42)
print(f"train: {len(tr)} photos | test: {len(te)} photos | stratified, every person appears in both")

🆘 <details><summary><b>Why <code>stratify=y</code>?</b></summary>
It forces every person to appear in both train and test in equal proportion. Without it, a random split might put *all* of someone's photos in the test set, and the model could never recognise them.
</details>

#### 🧬 Block 4: Eigenfaces explorer (4 min)

In Block 2 *we* chose 3 measurements. Here the **machine finds its own**. **Principal Component Analysis (PCA)** finds the directions in which faces vary most. The resulting patterns, drawn as images, are called **eigenfaces** (Turk &amp; Pentland, 1991).

Any face ≈ **average face + a weighted sum of eigenfaces**:
```python
 face  ≈  mean face  +  w₁·(eigenface 1)  +  w₂·(eigenface 2)  + …  +  w_k·(eigenface k)
```
The weights `w₁…w_k` are the face's **k numbers**, and that short vector is what gets compared. This is *compression*: 4,096 numbers → k numbers.

##### 🎛️ Explore
- Slide **k components** and watch the reconstruction sharpen.
- Pick which **eigenface** to display. Early ones capture big things (lighting from one side); later ones fine detail.

🔮 **Predict:** about how many components before the reconstruction looks recognisable? 5? 20? 100?

In [ ]:
@lru_cache(maxsize=8)
def pca_full(whiten=False):
    tr, _ = get_split(0.25, 42)
    return PCA(n_components=150, whiten=whiten, random_state=0).fit(X[tr])

def eigen_explorer(test_photo=0, k=20, eigenface=1):
    pca = pca_full(False); tr, te = get_split(0.25, 42)
    x = X[te[test_photo]][None, :]
    z = pca.transform(x); z_k = z.copy()
    z_k[:, k:] = 0          # keep only the first k weights
    recon = pca.inverse_transform(z_k)
    var = pca.explained_variance_ratio_.cumsum()
    fig, ax = plt.subplots(1, 4, figsize=(13, 3.2), gridspec_kw={"width_ratios": [1, 1, 1, 1.5]})
    ax[0].imshow(x.reshape(64, 64), cmap="gray")
    ax[0].set_title("original (4096 numbers)")
    ax[1].imshow(recon.reshape(64, 64), cmap="gray")
    ax[1].set_title(f"rebuilt from {k} numbers")
    ax[2].imshow(pca.components_[eigenface - 1].reshape(64, 64), cmap="gray"); ax[2].set_title(f"eigenface #{eigenface}")
    for a in ax[:3]: a.axis("off")
    ax[3].plot(range(1, 151), var, color="#6c3fc5"); ax[3].axvline(k, color="crimson", ls="--")
    ax[3].set_xlabel("components kept")
    ax[3].set_ylabel("variance retained"); ax[3].set_ylim(0, 1.01); ax[3].grid(alpha=.3)
    ax[3].set_title(f"{var[k-1]:.0%} of variation kept at k={k}")
    plt.tight_layout();
    plt.show()

W.interact(eigen_explorer,
    test_photo=W.IntSlider(0, min=0, max=99, description="📷 photo", continuous_update=False),
    k=W.IntSlider(20, min=1, max=150, description="🧮 k comps", continuous_update=False),
    eigenface=W.IntSlider(1, min=1, max=50, description="👻 eigenface", continuous_update=False));

<details><summary>✅ What you should notice</summary>

- With ~10 components the face is a blurry blob but the *gist* (hair, face shape, glasses) is there; by ~50 it is clearly the person; beyond ~100 you gain almost nothing.
- The variance curve flattens: most of the information lives in the first few dozen directions. That is why PCA can throw away >98% of the numbers and still work.
- ⚠️ "Variance retained" is not the same as "identity retained". The directions that vary most (lighting!) are not necessarily the ones that distinguish people. A theme for the debrief.
</details>

In [ ]:
quiz("Eigenfaces are best described as…",
     ["Photos of the most famous people in the dataset",
      "Directions of greatest variation across all training faces, shown as images",
      "Hand-drawn features such as eye distance",
      "The faces the model got wrong"], 1,
     "PCA finds the directions along which the training faces vary most; plotted as images they look like ghostly faces.")

#### Block 5: The hyperparameter lab (10 min)


| | Learned from data? | Examples here |
|---|---|---|
| **Parameters** | ✅ yes (the training finds them) | eigenfaces, the stored training vectors |
| **Hyperparameters** | ❌ no, **you** choose them | number of PCA components, *k* neighbours, distance metric, whitening, train/test ratio |

Choosing hyperparameters is a major part of real ML work, and the most common source of "my results can't be reproduced".

What each control does
| Control | What it changes | Risk at extremes |
|---|---|---|
| **Features** | raw 4,096 pixels *or* PCA vector | raw: slow, noisy. PCA: may discard useful detail |
| **PCA components** | length of the feature vector | too few → underfit (can't tell people apart), too many → keeps noise |
| **Whiten** | rescales every PCA direction to equal variance | can amplify noise in weak directions |
| ***k* neighbours** | how many nearest photos vote | k=1 sensitive to a single odd photo · large k blurs people together (only ~7 photos per person!) |
| **Metric** | how "distance" is measured | cosine ignores brightness scale, Manhattan is less outlier-sensitive |
| **Weights** | do closer neighbours get more say? | |
| **Test fraction** | how much data is held out | big test → less to learn from; small test → noisy estimate |
| **Seed** | which photos land in train vs test | different luck → different accuracy! |

🔮 Predict-then-test protocol (write in your lab log!)
Before touching a control, complete: *"If I **increase/decrease ___**, accuracy will **rise/fall/stay** because ___."* Then test it. Use **Log run** to record interesting settings.

In [ ]:
RUNS = []          # your lab log

def run_experiment(test_size=0.25, seed=42, use_pca=True, n_components=50, whiten=True,
                   k=1, metric="euclidean", weights="uniform", corrupt=None):
    t0 = time.perf_counter()
    tr, te = get_split(test_size, seed)
    Xtr, Xte = X[tr], X[te]
    if corrupt is not None: Xte = corrupt(Xte)
    if use_pca:
        pca = fit_pca(test_size, seed, n_components, whiten)
        Xtr, Xte = pca.transform(Xtr), pca.transform(Xte)
    knn = KNeighborsClassifier(n_neighbors=k, metric=metric, weights=weights).fit(Xtr, y[tr])
    pred = knn.predict(Xte)
    return dict(acc=accuracy_score(y[te], pred), pred=pred, tr=tr, te=te, knn=knn, Xte=Xte,
                dim=Xtr.shape[1], ms=(time.perf_counter() - t0) * 1000)

ctl = dict(
    use_pca=W.ToggleButtons(options=[("Raw pixels", False), ("Eigenfaces (PCA)", True)], value=True, description="Features"),
    n_components=W.IntSlider(50, min=2, max=150, description="PCA comps", continuous_update=False),
    whiten=W.Checkbox(True, description="whiten"),
    k=W.IntSlider(1, min=1, max=9, step=2, description="k neighbours", continuous_update=False),
    metric=W.Dropdown(options=["euclidean", "manhattan", "cosine"], value="euclidean", description="metric"),
    weights=W.ToggleButtons(options=["uniform", "distance"], description="vote"),
    test_size=W.Dropdown(options=[0.2, 0.25, 0.3, 0.4, 0.5], value=0.25, description="test frac"),
    seed=W.IntSlider(42, min=0, max=99, description="seed", continuous_update=False),
)

def show_run(**p):
    r = run_experiment(**p)
    wrong = np.where(r["pred"] != y[r["te"]])[0]
    color = "#2e8b57" if r["acc"] >= .9 else "#e6a700" if r["acc"] >= .7 else "#c0392b"
    display(HTML(f"<div style='font-size:20px'>accuracy <b style='color:{color}'>{r['acc']:.1%}</b> &nbsp;|&nbsp; ❌ {len(wrong)} / {len(r['te'])} wrong &nbsp;|&nbsp; {r['dim']} numbers per face &nbsp;|&nbsp; ⏱️ {r['ms']:.0f} ms</div>"))
    if len(wrong):
        _, nbr = r["knn"].kneighbors(r["Xte"][wrong[:8]], n_neighbors=1)
        n = min(8, len(wrong)); fig, ax = plt.subplots(2, n, figsize=(1.7 * n, 3.7), squeeze=False)
        for j in range(n):
            ax[0, j].imshow(X[r["te"][wrong[j]]].reshape(64, 64), cmap="gray")
            ax[0, j].set_title(f"true {y[r['te'][wrong[j]]]}", fontsize=8, color="#2e8b57")
            ax[1, j].imshow(X[r["tr"][nbr[j, 0]]].reshape(64, 64), cmap="gray")
            ax[1, j].set_title(f"guess {r['pred'][wrong[j]]}", fontsize=8, color="#c0392b")
        for a in ax.ravel(): a.axis("off")
        plt.suptitle("🔍 First mistakes: top = the new photo, bottom = who the model thought it was", y=1.02, fontsize=10); plt.tight_layout(); plt.show()
    else:
        print("🎉 No mistakes. Try making the problem harder (fewer components, bigger test fraction).")

log_btn = W.Button(description="Log run", button_style="success")
clear_btn = W.Button(description="🗑️ Clear log", button_style="warning")
log_out = W.Output()

def current():
    return {k_: w.value for k_, w in ctl.items()}

def refresh_log():
    with log_out:
        clear_output(wait=True)
        if not RUNS: print("Log is empty. Tweak the controls and press Log run."); return
        print(f"{'#':>2} {'features':8s} {'comps':>5} {'whiten':>6} {'k':>2} {'metric':10s} {'vote':8s} {'test':>5} {'seed':>4}   accuracy")
        for i, (p, a) in enumerate(RUNS, 1):
            print(f"{i:>2} {('PCA' if p['use_pca'] else 'raw'):8s} {p['n_components'] if p['use_pca'] else '-':>5} {str(p['whiten'] if p['use_pca'] else '-'):>6} {p['k']:>2} {p['metric']:10s} {p['weights']:8s} {p['test_size']:>5} {p['seed']:>4}   {a:.1%}")
        fig, ax = plt.subplots(figsize=(7, 0.35 * len(RUNS) + 1.2))
        ax.barh([f"run {i}" for i in range(1, len(RUNS) + 1)], [a for _, a in RUNS], color="#6c3fc5"); ax.invert_yaxis()
        ax.set_xlim(0, 1); ax.set_xlabel("accuracy"); ax.set_title("📊 Your runs compared"); plt.tight_layout(); plt.show()

def on_log(_):
    p = current(); RUNS.append((p, run_experiment(**p)["acc"])); refresh_log()
def on_clear(_):
    RUNS.clear(); refresh_log()
log_btn.on_click(on_log); clear_btn.on_click(on_clear)

left = W.VBox([ctl["use_pca"], ctl["n_components"], ctl["whiten"], ctl["k"]])
right = W.VBox([ctl["metric"], ctl["weights"], ctl["test_size"], ctl["seed"]])
display(W.HBox([left, right]))
display(W.HBox([log_btn, clear_btn]))
display(W.interactive_output(show_run, ctl))
display(W.HTML("<h4>Lab log</h4>")); display(log_out); refresh_log()

### 🔬 Experiments to run (pick 3, log each one)
| # | 🔮 Question | Try |
|---|---|---|
| 1 | Is PCA better than raw pixels, and is it **faster**? | Toggle *Features*; watch accuracy **and** ms |
| 2 | How few components are enough? | PCA comps = 2, 5, 10, 25, 50 |
| 3 | Does whitening help? | Tick/untick with 50 and then with 150 components |
| 4 | Is the nearest single photo best? | k = 1, 3, 5, 9 |
| 5 | Which metric wins? | euclidean / manhattan / cosine |
| 6 | How much does luck matter? | Keep everything fixed; change only **seed** 5 times |
| 7 | Does less training data hurt? | test fraction 0.2 → 0.5 |

<details><summary>🆘 Hint for #2 and #4</summary>

- #2: accuracy usually climbs steeply up to ~20–30 components, then plateaus. Beyond that, extra components mostly add noise.
- #4: with only ~7 training photos per person, large *k* forces the vote to include other people. Expect accuracy to *drop* as k grows; **k=1 is often best here**. This depends on the dataset!
</details>

<details><summary>✅ Typical findings (your numbers will differ slightly)</summary>

- 🧬 PCA ≈ raw-pixel accuracy but with ~80× fewer numbers, so it is faster and more compact.
- 🎲 Changing only the seed moves accuracy by a few points. **One run is an anecdote**; report mean ± spread (Block 6).
- ⚪ Whitening often helps a little at moderate sizes and can hurt at very large sizes (amplified noise).
</details>

In [ ]:
quiz("You change only the random seed and accuracy moves from 93% to 96%. The right conclusion is…",
     ["The model improved", "The split changed, so there is natural run-to-run variation", "The dataset is corrupted", "PCA is broken"], 1,
     "Different seeds create different train/test splits. A few points of movement is sampling noise, so compare methods over several seeds.")

#### Block 6: Sweeps, variance and reproducibility 🚀

Sliders are for *exploring*. For a claim you can defend you need to: (1) **sweep** systematically, (2) **repeat over seeds**, (3) report **mean ± standard deviation**. Below is a systematic grid over *k* and PCA size.

In [ ]:
KS = [1, 3, 5, 7, 9]
NCS = [5, 10, 20, 35, 50, 75, 100, 150]
SEEDS = [1, 2, 3]
grid = np.zeros((len(KS), len(NCS)))
for i, kk in enumerate(KS):
    for j, nc in enumerate(NCS):
        grid[i, j] = np.mean([run_experiment(seed=s, n_components=nc, k=kk)["acc"] for s in SEEDS])

fig, ax = plt.subplots(figsize=(8.5, 3.6))
im = ax.imshow(grid, cmap="viridis", aspect="auto", vmin=max(0, grid.min()), vmax=1)
ax.set_xticks(range(len(NCS)), NCS)
ax.set_yticks(range(len(KS)), KS)
ax.set_xlabel("PCA components")
ax.set_ylabel("k neighbours")
ax.set_title(f"Mean accuracy over {len(SEEDS)} seeds")
for i in range(len(KS)):
    for j in range(len(NCS)):
      ax.text(j, i, f"{grid[i, j]:.0%}", ha="center", va="center", color="white" if grid[i, j] < .85 else "black", fontsize=8)
plt.colorbar(im, label="accuracy"); plt.tight_layout(); plt.show()
bi, bj = np.unravel_index(grid.argmax(), grid.shape)
print(f"🏆 best cell: k={KS[bi]}, components={NCS[bj]} → {grid[bi, bj]:.1%}")

#### How big is the luck factor?
Same model, **30 different splits**. This histogram is the honest answer to "what is the accuracy?"

In [ ]:
accs = np.array([run_experiment(seed=s)["acc"] for s in range(30)])
fig, ax = plt.subplots(figsize=(7, 3)); ax.hist(accs, bins=12, color="#6c3fc5", edgecolor="white")
ax.axvline(accs.mean(), color="crimson", ls="--", label=f"mean {accs.mean():.1%}"); ax.legend()
ax.set_xlabel("accuracy"); ax.set_ylabel("# of splits"); ax.set_title("Accuracy over 30 random splits"); plt.tight_layout(); plt.show()
print(f"📊 report this as:  {accs.mean():.1%} ± {accs.std():.1%}   (min {accs.min():.1%}, max {accs.max():.1%})")

🤔 **Discuss:** two students compare methods A and B using one seed each: A gets 94%, B gets 92%. Can they claim A is better? What would you do instead?

<details><summary>✅ Answer</summary>
Not on that evidence: the gap is within split-to-split noise. Repeat over many seeds (or use k-fold cross-validation), compare means ± spread, ideally with a paired test. Also **fix and report the seed** so others can reproduce your numbers.
</details>

#### ✅ Reproducibility checklist (take this home)
- [ ] Fixed and reported random seeds
- [ ] Reported library versions (Block 0 printout)
- [ ] Held-out test set never used for tuning
- [ ] Mean ± spread over several splits
- [ ] Hyperparameters listed
- [ ] Limitations of the dataset stated

### Block 7: Robustness: noise, masks, lighting 🚀

A model is evaluated on photos *like its training photos*. Real deployments are messier: a different camera (**noise**), a face mask or scarf (**occlusion**), a darker room (**brightness**). When test data differs from training data, performance can collapse silently. This is called **distribution shift**.

#### 🎛️ Corrupt the *test* photos only (training stays clean)
🔮 **Predict:** which will hurt more, 30% noise or a mask covering the lower 30% of the face? Does PCA make the model more or less robust than raw pixels?

In [ ]:
def corrupt_images(Xte, noise=0.0, occlude=0.0, brightness=0.0, seed=0):
    imgs = Xte.reshape(-1, 64, 64).copy()
    rng = np.random.RandomState(seed)
    imgs = imgs + brightness + noise * rng.randn(*imgs.shape)
    if occlude > 0: imgs[:, 64 - int(64 * occlude):, :] = 0.0      # black "mask" over the lower face
    return np.clip(imgs, 0, 1).reshape(len(imgs), -1)

def robustness(noise=0.0, occlude=0.0, brightness=0.0, use_pca=True, n_components=50):
    cfn = lambda Xt: corrupt_images(Xt, noise, occlude, brightness)
    r = run_experiment(use_pca=use_pca, n_components=n_components, corrupt=cfn)
    base = run_experiment(use_pca=use_pca, n_components=n_components)["acc"]
    fig, ax = plt.subplots(1, 6, figsize=(12, 2.3))
    for a, i in zip(ax, range(6)): a.imshow(cfn(X[r["te"][i:i+1]]).reshape(64, 64), cmap="gray", vmin=0, vmax=1); a.axis("off")
    plt.suptitle("What the camera now delivers (test photos)", y=1.03); plt.show()
    drop = base - r["acc"]
    display(HTML(f"<div style='font-size:19px'>clean accuracy {base:.1%} → corrupted <b style='color:{'#c0392b' if drop>.1 else '#2e8b57'}'>{r['acc']:.1%}</b> &nbsp;(drop {drop:+.1%})</div>"))

W.interact(robustness,
    noise=W.FloatSlider(0.0, min=0, max=0.6, step=0.05, description="noise", continuous_update=False),
    occlude=W.FloatSlider(0.0, min=0, max=0.6, step=0.05, description="mask %", continuous_update=False),
    brightness=W.FloatSlider(0.0, min=-0.3, max=0.3, step=0.05, description="brightness", continuous_update=False),
    use_pca=W.ToggleButtons(options=[("Raw", False), ("PCA", True)], value=True, description="Features"),
    n_components=W.IntSlider(50, min=5, max=150, description="comps", continuous_update=False));

In [ ]:
# 📉 Systematic version: accuracy vs mask size, raw pixels vs PCA
levels = np.arange(0, 0.65, 0.1)
raw = [run_experiment(use_pca=False, corrupt=lambda Xt, o=o: corrupt_images(Xt, occlude=o))["acc"] for o in levels]
pc  = [run_experiment(use_pca=True,  corrupt=lambda Xt, o=o: corrupt_images(Xt, occlude=o))["acc"] for o in levels]
plt.figure(figsize=(6.5, 3.4)); plt.plot(levels, raw, "o-", label="raw pixels"); plt.plot(levels, pc, "s-", label="PCA (50)")
plt.xlabel("fraction of lower face covered"); plt.ylabel("test accuracy"); plt.ylim(0, 1.02); plt.grid(alpha=.3); plt.legend(); plt.title("Robustness to occlusion"); plt.tight_layout(); plt.show()

<details><summary>✅ Discussion points</summary>

- Both representations degrade as the mask grows; the model was never shown masked faces, so nothing in training prepares it. *Data coverage* matters more than algorithm choice.
- Brightness shifts can hurt PCA a lot because the early eigenfaces encode lighting.
- 🛠️ Real mitigations: train on augmented/masked data, use learned deep features, add a **confidence threshold** and route low-confidence cases to a human.
</details>

### Block 8: Strangers, thresholds, FAR vs FRR (7 min)

Until now every test person was enrolled ("closed set"). Reality: **most people who walk past a camera are strangers**. The system must say *"unknown"*, which brings back the **threshold** from Block 2.

We hide a few people from training entirely (**strangers**) and measure each probe's distance to its *nearest enrolled photo*.

| Error | Meaning | Who gets hurt |
|---|---|---|
| ❌ **False Reject Rate (FRR)** | an enrolled person is turned away | inconvenience: "my phone won't open" |
| ⚠️ **False Accept Rate (FAR)** | a stranger is let in | security breach or a **wrongful match** |

Lowering the threshold trades FAR for FRR. **The point where they are equal is the Equal Error Rate (EER)**, a common single-number summary, but *which trade-off is acceptable is an ethical decision, not a technical one.*

### 🎛️ Controls
- 🕵️ **strangers**: how many people are hidden from the system
- **accept the closest X% of enrolled probes**: slide right to be more permissive
- **PCA comps**: a better representation should separate the two histograms

🔮 **Predict:** if you accept 100% of enrolled probes (FRR = 0), roughly how many strangers get in?

In [ ]:
@lru_cache(maxsize=64)
def open_set_distances(n_strangers, n_components, seed=42):
    rng = np.random.RandomState(seed)
    stranger_ids = rng.choice(np.unique(y), n_strangers, replace=False)
    is_str = np.isin(y, stranger_ids)
    Xk, yk, Xs = X[~is_str], y[~is_str], X[is_str]
    Xtr, Xte, _, _ = train_test_split(Xk, yk, test_size=0.25, stratify=yk, random_state=seed)
    pca = PCA(n_components=n_components, whiten=True, random_state=0).fit(Xtr)
    nn = NearestNeighbors(n_neighbors=1).fit(pca.transform(Xtr))
    dk = nn.kneighbors(pca.transform(Xte))[0].ravel(); ds = nn.kneighbors(pca.transform(Xs))[0].ravel()
    return dk, ds

def open_set_lab(n_strangers=5, n_components=50, accept_pct=90):
    dk, ds = open_set_distances(n_strangers, n_components)
    thr = np.percentile(dk, accept_pct)
    frr, far = (dk > thr).mean(), (ds <= thr).mean()
    ts = np.linspace(min(dk.min(), ds.min()), max(dk.max(), ds.max()), 300)
    FRR = np.array([(dk > t).mean() for t in ts]); FAR = np.array([(ds <= t).mean() for t in ts])
    e = np.argmin(np.abs(FRR - FAR)); eer = (FRR[e] + FAR[e]) / 2
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    bins = np.linspace(ts.min(), ts.max(), 30)
    ax[0].hist(dk, bins, alpha=.7, label="✅ enrolled people", color="#2e8b57"); ax[0].hist(ds, bins, alpha=.7, label="🕵️ strangers", color="#c0392b")
    ax[0].axvline(thr, color="k", ls="--", label=f"threshold {thr:.1f}"); ax[0].set_xlabel("distance to nearest enrolled photo"); ax[0].legend(); ax[0].set_title("📊 Do the two groups separate?")
    ax[1].plot(ts, FRR, label="FRR (enrolled rejected)"); ax[1].plot(ts, FAR, label="FAR (stranger accepted)")
    ax[1].axvline(thr, color="k", ls="--"); ax[1].scatter([ts[e]], [eer], color="gold", edgecolor="k", zorder=5, s=90, label=f"EER ≈ {eer:.1%}")
    ax[1].set_xlabel("threshold"); ax[1].set_ylabel("rate"); ax[1].legend(); ax[1].grid(alpha=.3); ax[1].set_title("The FAR / FRR trade-off")
    plt.tight_layout(); plt.show()
    display(HTML(f"<div style='font-size:19px'>FRR <b>{frr:.1%}</b> &nbsp;|&nbsp; FAR <b>{far:.1%}</b> &nbsp;|&nbsp; EER ≈ {eer:.1%} &nbsp;|&nbsp; strangers tested: {len(ds)} photos</div>"))

W.interact(open_set_lab,
    n_strangers=W.IntSlider(5, min=3, max=10, description="🕵️ strangers", continuous_update=False),
    n_components=W.IntSlider(50, min=5, max=150, description="PCA comps", continuous_update=False),
    accept_pct=W.FloatSlider(90, min=50, max=100, step=1, description="accept %", continuous_update=False));

### 🤔 Discuss
For each deployment, which error is worse, and where would you set the threshold (low = strict, high = permissive)?
| Deployment | Worse error? | Threshold |
|---|---|---|
| 📱 unlocking your phone | | |
| 🏦 approving a bank transfer | | |
| 🚓 police matching a suspect photo to a database | | |
| 🎓 taking classroom attendance | | |

<details><summary>✅ Suggested reasoning</summary>

- Phone: FAR is a privacy risk but you can retry (low FRR cost) → fairly strict.
- Bank: a false accept is costly → strict, plus a second factor.
- Police: a **false accept means an innocent person may be treated as a suspect**, with documented real-world wrongful arrests. Strict threshold *and* mandatory human review; many argue the use itself needs strong safeguards.
- Attendance: low stakes either way → permissive, human correctable.

The maths gives the curve; **people choose the point on it**.
</details>

In [ ]:
quiz("Lowering the distance threshold (making the system stricter) will usually…",
     ["Raise FAR and lower FRR", "Lower FAR and raise FRR", "Lower both", "Raise both"], 1,
     "A stricter threshold accepts fewer people: fewer strangers get in (FAR down) but more genuine users are turned away (FRR up).")

### Block 9: Per-person errors, bias and ethics 🚀

Overall accuracy can look excellent while the model **fails some individuals far more than others**. In real systems, researchers have shown error rates can differ strongly across demographic groups (see *Gender Shades* in the references). Auditing means **breaking results down by group**.

Below: accuracy per person across 30 random splits. (Olivetti has no demographic labels, so we can only see *individual* differences, but the habit is the same.)

In [ ]:
per_person = np.zeros(40); counts = np.zeros(40)
for s in range(30):
    r = run_experiment(seed=s); yt = y[r["te"]]
    for p_true, p_pred in zip(yt, r["pred"]):
        counts[p_true] += 1; per_person[p_true] += (p_true == p_pred)
acc_p = per_person / counts; order = np.argsort(acc_p)
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(range(40), acc_p[order], color=["#c0392b" if a < .9 else "#2e8b57" for a in acc_p[order]])
ax.set_xticks(range(40), order, fontsize=7); ax.set_xlabel("person id (sorted worst → best)"); ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.05)
ax.set_title(f"🔎 Same model, very different experiences. Overall mean {acc_p.mean():.1%}, worst person {acc_p.min():.0%}"); plt.tight_layout(); plt.show()
w = order[:4]
fig, ax = plt.subplots(1, 4, figsize=(8, 2.2))
for a, p in zip(ax, w): a.imshow(IMG[np.where(y == p)[0][0]], cmap="gray"); a.set_title(f"person {p}: {acc_p[p]:.0%}", fontsize=9); a.axis("off")
plt.suptitle("The four hardest people for this model", y=1.04); plt.show()

### 🤔 Research ethics panel (10 min, 4 groups)
Assign one role to each group, then debate: **"A university wants to use face recognition to take lecture attendance."**
| 🎭 Role | Your job |
|---|---|
| 🧑‍🎓 **Student** | privacy, consent, opting out |
| 🧑‍🏫 **Lecturer** | accuracy needs, fairness, workload |
| 🛡️ **Data-protection officer** | lawful basis, data retention, security of face templates |
| 🔬 **ML researcher** | what must be measured before deployment? (metrics per group, robustness, monitoring) |

**Questions to resolve:** Who consents? Where are templates stored, and can a face be "changed" like a password? What happens on a false reject? Who audits the model, and how often?

> 🌍 Facial recognition is regulated or restricted in various jurisdictions. Check the rules that apply where you work.

### Block 10: Debrief and research discussion (4 min)

| Stage | In this lab |
|---|---|
| 1️⃣ Problem and data | Faces, enrolment, Olivetti dataset (Blocks 1, 3) |
| 2️⃣ Preparation | Train/test split, stratification (Block 3) |
| 3️⃣ Modelling | PCA + nearest neighbour; hyperparameters (Blocks 4, 5) |
| 4️⃣ Evaluation | Accuracy, FAR/FRR/EER, seeds, per-person (Blocks 6, 8, 9) |
| 5️⃣ Deployment risks | Robustness, thresholds, ethics (Blocks 7, 8, 9) |
| 6️⃣ Monitoring | *What would drift after launch?* (discuss) |

### Philosophy: three questions to close with
1. 🤖 vs 🧑 **Rational or human-like?** The recogniser minimises a distance to optimise a performance measure. Does it behave the way *a person* recognises a friend (context, voice, memory, a feeling of familiarity)? Would an *optimal* recogniser even need to?
2. **Chinese Room.** Searle argues that following symbol-manipulation rules is not understanding. Our recogniser matches number vectors. Is there anything it "knows" about faces or persons? Does it matter if the behaviour is useful?
3. **Turing test for faces?** If a system matched humans' accuracy on this dataset, would you call it "human-like"? What if it was *better* than humans but made *different* mistakes?

<details><summary>💡 Pointers for the discussion</summary>

- *Rational* (Russell &amp; Norvig): does "the right thing" given its objective, here minimising recognition error. *Human-like*: reproduces human cognitive processes or behaviour, including human errors and biases. Eigenfaces are rational-engineering, not cognitive modelling.
- Simon's **bounded rationality**: real agents (including us) satisfice under limited time and information. Our threshold is a satisficing rule, not a proof of optimality.
- Searle: syntax alone does not give semantics. A defender of the "systems reply" would say understanding might emerge from the whole system; our toy is far too simple for either side to claim much.
</details>

### 🏁 Take-home challenges (pick one)
1. Add `SVC(kernel="linear")` or `LogisticRegression` and compare with k-NN over 10 seeds (mean ± std).
2. Add a **"train with masked faces"** step and measure whether masked-face accuracy recovers.
3. Plot a full **ROC curve** (FAR vs 1-FRR) and compute its area.
4. Write a 150-word *model card*: intended use, data, metrics, limitations, ethical concerns.

### 📚 Further reading
- Turk, M. &amp; Pentland, A. (1991). *Eigenfaces for Recognition.* Journal of Cognitive Neuroscience, 3(1), 71-86. DOI: 10.1162/jocn.1991.3.1.71
- Buolamwini, J. &amp; Gebru, T. (2018). *Gender Shades.* PMLR 81: http://proceedings.mlr.press/v81/buolamwini18a.html
- NIST Face Recognition Vendor Test (FRVT): https://www.nist.gov/programs-projects/face-recognition-vendor-test-frvt
- Russell &amp; Norvig, *Artificial Intelligence: A Modern Approach*, Ch. 1 and Ch. 2 (rational agents, PEAS)
- Searle (1980), "Minds, Brains, and Programs"; Stanford Encyclopedia of Philosophy: https://plato.stanford.edu/entries/chinese-room/
- Turing (1950), *Computing Machinery and Intelligence.* Mind 59(236)
- scikit-learn user guide (PCA, neighbours, model evaluation): https://scikit-learn.org/stable/user_guide.html


### Glossary
| Term | Meaning |
|---|---|
| **Enrolment / gallery** | the stored known faces |
| **Probe** | the new face being checked |
| **Feature / embedding** | the numbers that describe a face |
| **PCA / eigenfaces** | learned directions of greatest variation |
| **k-NN** | classify by the *k* nearest stored examples |
| **Hyperparameter** | a setting chosen by the human, not learned |
| **Threshold** | maximum distance still accepted as a match |
| **FAR / FRR / EER** | stranger-accepted rate / enrolled-rejected rate / where they are equal |
| **Open-set** | the system must also recognise "none of the above" |
| **Distribution shift** | deployment data differs from training data |
| **Overfitting** | doing well on training data, poorly on new data |

<div style="text-align:center;font-size:18px">🎉 <b>Thank you for taking part!</b> 🎉<br><span style="font-size:14px">Save your lab log, jot one finding you were surprised by, and share it with a neighbour.</span></div>